# Save shallow-learning classification models + scaler (.pkl)

Trains the three shallow classifiers from `04_SL_classification.ipynb` (Logistic Regression, XGBoost, Random Forest) on the Barcelona dataset and saves each one, together with the `StandardScaler` and the feature column order, as pickle files in `models/`.

**Target: `noise_day`.** Saving the scaler alongside the models is essential: without it, new input data (e.g. Milan / Viladecans / Berlin) cannot be transformed into the space the models were trained on.

# Data and libraries

In [1]:
# standard packaging

import pandas as pd
import numpy as np
import pickle
import os

In [2]:
# load the dataset - original source

data = pd.read_csv("data/bcn_noise_class_ml_dataset.csv")

In [3]:
data_clean = data.dropna(how = "any")
data_clean.reset_index(inplace=True, drop = True)
data_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 12854 entries, 0 to 12853
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   road_id                12854 non-null  str    
 1   noise_day              12854 non-null  int64  
 2   noise_evening          12854 non-null  int64  
 3   noise_night            12854 non-null  int64  
 4   road_category          12854 non-null  int64  
 5   dist_to_trunk          12854 non-null  float64
 6   dist_to_primary        12854 non-null  float64
 7   dist_to_secondary      12854 non-null  float64
 8   dist_to_tertiary       12854 non-null  float64
 9   dist_to_residential    12854 non-null  float64
 10  dist_to_living_street  12854 non-null  float64
 11  signals                12854 non-null  float64
 12  transport              12854 non-null  float64
 13  pois                   12854 non-null  float64
 14  width                  12854 non-null  float64
 15  betweenness  

### scaling

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [5]:
X = data_clean.drop(columns=['road_id','noise_day','noise_evening', 'noise_night']) 
X_scaled = scaler.fit_transform(X)
X.head()

,road_category,dist_to_trunk,dist_to_primary,dist_to_secondary,dist_to_tertiary,dist_to_residential,dist_to_living_street,signals,transport,pois,width,betweenness,closeness_global,closeness_400,straightness,green,industrial,commercial
0,5,213.093233,512.208689,1110.052349,188.355179,0.0,385.612798,0.053298,0.000000,0.106597,50.000000,0.000039,0.000187,0.000017,0.807756,3.349277,0.0,0.0
1,5,549.529452,1655.791584,2110.530434,351.777378,0.0,117.669324,0.000000,0.014274,0.000000,20.006115,0.000224,0.000109,0.000004,0.741368,0.000000,0.0,0.0
2,3,904.118521,947.154817,0.000000,167.026629,0.0,19.810222,0.167307,0.000000,0.295484,50.000000,0.010452,0.000182,0.000018,0.831327,8.619238,0.0,0.0
3,5,1877.603526,419.287598,661.587375,10.597441,0.0,41.365655,0.000000,0.012453,0.002491,16.759462,0.004427,0.000215,0.000009,0.757172,4.346695,0.0,0.0
4,5,1872.177506,614.353897,967.746460,0.000000,0.0,49.109562,0.000000,0.053513,0.000000,50.000000,0.003735,0.000208,0.000006,0.736805,25.169641,0.0,0.0


In [6]:
y = np.asarray(data_clean["noise_day"])
y

array([2, 2, 3, ..., 4, 2, 2], shape=(12854,))

### splitting

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state = 42)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

(10283, 18)
(10283,)
(2571, 18)


# Save scaler and feature column order

In [8]:
# output folder for all pickled artifacts
os.makedirs('models', exist_ok=True)

In [9]:
#SAVE SCALER FOR LATER USE

with open('models/Sscaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [10]:
#save the feature column order - needed to arrange a new city's columns before scaler.transform

feature_columns = list(X.columns)
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

feature_columns

['road_category',
 'dist_to_trunk',
 'dist_to_primary',
 'dist_to_secondary',
 'dist_to_tertiary',
 'dist_to_residential',
 'dist_to_living_street',
 'signals',
 'transport',
 'pois',
 'width',
 'betweenness',
 'closeness_global',
 'closeness_400',
 'straightness',
 'green',
 'industrial',
 'commercial']

# Train and save models

## Logistic Regression

In [11]:
from sklearn.linear_model import LogisticRegression
logreg_model = LogisticRegression(max_iter=2000)

logreg_model.fit(X_train, y_train)
print(logreg_model.score(X_train, y_train))
print(logreg_model.score(X_test, y_test))

0.617232325196927
0.6063788409179308


In [12]:
#save sk learn model

with open('models/logreg_class_model.pkl', 'wb') as f:
    pickle.dump(logreg_model, f)

## XG boost

In [13]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    )

xgb_model.fit(X_train, y_train)
print(xgb_model.score(X_train, y_train))
print(xgb_model.score(X_test, y_test))

0.9717008655061753
0.7405678724231817


In [14]:
#save xgboost model

with open('models/xgb_class_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

## Random Forest

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=42,
    max_depth=None,         
    min_samples_leaf=1,    
    min_samples_split=2,   
)

rf_model.fit(X_train, y_train)
print(rf_model.score(X_train, y_train))
print(rf_model.score(X_test, y_test))

0.9984440338422639
0.7487359004278491


In [16]:
#save sk learn model

with open('models/rf_class_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Verify the saved files

Reload every pickle and check the loaded models reproduce the in-memory test accuracy.

In [17]:
with open('models/Sscaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
with open('models/feature_columns.pkl', 'rb') as f:
    loaded_columns = pickle.load(f)

# transform the raw (unscaled) features with the loaded scaler
X_check = loaded_scaler.transform(data_clean[loaded_columns])
_, X_check_test, _, y_check_test = train_test_split(X_check, y, test_size = 0.2, random_state = 42)

for name, path, trained in [
    ('Logistic Regression', 'models/logreg_class_model.pkl', logreg_model),
    ('XGBoost',             'models/xgb_class_model.pkl',    xgb_model),
    ('Random Forest',       'models/rf_class_model.pkl',     rf_model),
]:
    with open(path, 'rb') as f:
        loaded_model = pickle.load(f)
    acc_loaded = loaded_model.score(X_check_test, y_check_test)
    acc_memory = trained.score(X_test, y_test)
    assert abs(acc_loaded - acc_memory) < 1e-9, name
    print(f"{name}: loaded test accuracy = {acc_loaded:.4f}  (matches in-memory model)")

Logistic Regression: loaded test accuracy = 0.6064  (matches in-memory model)


XGBoost: loaded test accuracy = 0.7406  (matches in-memory model)
Random Forest: loaded test accuracy = 0.7487  (matches in-memory model)


In [18]:
# demo: predict noise_day class for 5 rows, as a deployment would

demo_raw = data_clean[loaded_columns].sample(5, random_state=0)
demo_scaled = loaded_scaler.transform(demo_raw)

with open('models/rf_class_model.pkl', 'rb') as f:
    rf_loaded = pickle.load(f)

pd.DataFrame({
    'road_id': data_clean.loc[demo_raw.index, 'road_id'].values,
    'noise_day_true': data_clean.loc[demo_raw.index, 'noise_day'].values,
    'noise_day_pred': rf_loaded.predict(demo_scaled),
})

,road_id,noise_day_true,noise_day_pred
0,30295888_30295890_0,2,2
1,30554956_30554852_0,2,2
2,1303465752_1303465757_0,3,3
3,1378060299_1378060258_0,3,3
4,1350466016_1350466211_0,1,1
